## 基础练习

1. 输入一组 JD 技能词，统计词频并按次数降序输出。
2. 读取 JSON 配置，缺少 `model` 时给出清晰错误。
3. 定义 `StudyTask` Pydantic 模型：标题、优先级、预计小时、是否完成。
4. 将任务列表保存为 UTF-8 JSON，再读取并计算总小时。
5. 写异步函数模拟三个模型请求，比较串行与并发耗时。

In [65]:
# 1. 输入一组 JD 技能词，统计词频并按次数降序输出。
# 统计词频
skills = ["Python", "SQL", "Python", "RAG", "SQL", "Python"]

counts={}
for skill in skills:
  if skill in counts:
    counts[skill]+=1
  else:
    counts[skill]=1

print(counts)

# 按次数降序
sorted_counts = sorted(
  counts.items(),
  key=lambda item:item[1],
  reverse=True
)

for skill,count in sorted_counts:
  print(f"{skill}:{count}")

{'Python': 3, 'SQL': 2, 'RAG': 1}
Python:3
SQL:2
RAG:1


### 2. 读取 JSON 配置

读取一个 JSON 配置文件，并检查配置中是否包含必填字段 `model`。

**要求：**

1. 使用 UTF-8 编码读取 JSON 文件。
2. 将 JSON 内容转换为 Python 字典。
3. 如果配置中存在 `model`，返回完整配置。
4. 如果缺少 `model`，抛出清晰的错误：`配置缺少必填字段：model`。
5. 分别使用一个正常配置和一个缺少 `model` 的配置进行验证。

In [66]:
import sys
from pathlib import Path
import json

valid_config = {
  'model':'kimi-k3',
  'temperature':0.2,
}

invalid_config = {
  'temperature':0.2,
}

Path('valid_config.json').write_text(
  json.dumps(valid_config,ensure_ascii=False, indent=2),
  encoding='utf-8'
)

Path('invalid_config.json').write_text(
  json.dumps(invalid_config,ensure_ascii=False, indent=2),
  encoding='utf-8'
)


24

In [67]:
def load_config(path:str) -> dict:
    config = json.loads(Path(path).read_text(encoding='utf-8'))

    if 'model' not in config:
        raise ValueError("配置缺少必填字段：model")
    return config

valid_config = load_config('valid_config.json')
print(valid_config)



try:
    invalid_config = load_config('invalid_config.json')
    print(invalid_config)
except ValueError as exc:
    print('预期中的校验错误:')
    print(exc)


{'model': 'kimi-k3', 'temperature': 0.2}
预期中的校验错误:
配置缺少必填字段：model


### 3. 定义 `StudyTask` Pydantic 模型

使用 Pydantic 定义一个学习任务模型 `StudyTask`，将外部输入转换成结构清晰、能够自动校验的 Python 对象。

**字段与校验要求：**

1. `title`：标题，字符串，长度为 1～100。
2. `priority`：优先级，整数，取值范围为 0～3。
3. `hours`：预计小时，浮点数，必须大于 0 且不超过 100。
4. `completed`：是否完成，布尔值，默认为 `False`。

**验证要求：**

- 创建一个合法任务，并使用 `model_dump()` 输出字典。
- 尝试创建一个标题为空、优先级越界或小时数不合法的任务。
- 捕获 `ValidationError`，并输出校验错误。

In [68]:
from pydantic import BaseModel, Field, ValidationError

class StudyTask(BaseModel):
    title: str = Field(min_length=1, max_length=100)
    priority: int = Field(ge=0, le=3)
    hours: float = Field(gt=0, le=100)
    completed: bool = Field(default=False)

task = StudyTask(title='学习 Python', priority=1, hours=8)
print(task.model_dump())
task.completed = True
print(task.model_dump())

try:
    StudyTask(title='', priority=9, hours=-1)
except ValidationError as exc:
    print('预期中的校验错误:')
    print(exc)

{'title': '学习 Python', 'priority': 1, 'hours': 8.0, 'completed': False}
{'title': '学习 Python', 'priority': 1, 'hours': 8.0, 'completed': True}
预期中的校验错误:
3 validation errors for StudyTask
title
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short
priority
  Input should be less than or equal to 3 [type=less_than_equal, input_value=9, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
hours
  Input should be greater than 0 [type=greater_than, input_value=-1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than


### 4. 保存并读取任务列表

创建一组 `StudyTask` 任务，将它们保存为 UTF-8 编码的 JSON 文件，然后重新读取文件并计算所有任务的预计总小时。

**要求：**

1. 创建至少 3 个 `StudyTask` 对象，并放入列表。
2. 使用 `model_dump()` 将每个 Pydantic 对象转换为字典。
3. 使用 `json.dumps()` 和 `Path.write_text()` 将列表写入 `tasks.json`。
4. 文件使用 UTF-8 编码，中文保持可读，JSON 需要格式化缩进。
5. 重新读取 `tasks.json`，并将每条数据恢复成 `StudyTask` 对象。
6. 使用 `sum()` 计算所有任务的 `hours` 总和并输出。

In [69]:
tasks = [StudyTask(title='学习 Python', priority=1, hours=8),
         StudyTask(title='学习 Pydantic', priority=2, hours=8),
         StudyTask(title='学习 json', priority=3, hours=8),]
tasks_data = []
for task in tasks:
     tasks_data.append(task.model_dump())

Path('tasks.json').write_text(
    json.dumps(tasks_data, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

loaded_data = json.loads(Path('tasks.json').read_text(encoding='utf-8'))
tasks_loaded= [
     StudyTask.model_validate(item)
     for item in loaded_data
               ]

total_hours  = sum(
     item.hours
     for item in tasks_loaded
)

print(total_hours)

24.0


### 5. 比较串行与并发模型请求

使用异步函数模拟 3 个需要等待的模型请求，分别以串行和并发方式执行，并比较两种方式的总耗时。

**要求：**

1. 定义 `async def fake_model_call(name: str, delay: float)` 异步函数。
2. 使用 `await asyncio.sleep(delay)` 模拟网络等待，不要使用 `time.sleep()`。
3. 串行执行 3 个请求，并用 `time.perf_counter()` 计时。
4. 使用 `asyncio.gather()` 并发执行同样的 3 个请求，并计时。
5. 输出两次的返回结果和耗时，观察串行与并发的差异。
6. 在 Notebook 中直接使用 `await` 调用，不需要使用 `asyncio.run()`。

In [70]:
# 定义模拟请求
import asyncio
import time


async def fake_model_call(name: str, delay: float) -> str:
    print(f"{name} 开始请求")
    await asyncio.sleep(delay)
    print(f"{name} 请求完成")
    return f"{name} 的结果"

In [71]:
# 串行执行
async def run_serial() -> list[str]:
    results = []

    result_a = await fake_model_call("模型 A", 1)
    results.append(result_a)

    result_b = await fake_model_call("模型 B", 1)
    results.append(result_b)

    result_c = await fake_model_call("模型 C", 1)
    results.append(result_c)

    return results

In [72]:
# 并发执行
async def run_concurrent() -> list[str]:
    results = await asyncio.gather(
        fake_model_call("模型 A", 1),
        fake_model_call("模型 B", 1),
        fake_model_call("模型 C", 1),
    )

    return results

In [73]:
started = time.perf_counter()
serial_results = await run_serial()
serial_elapsed = time.perf_counter() - started

started = time.perf_counter()
concurrent_results = await run_concurrent()
concurrent_elapsed = time.perf_counter() - started

print("串行结果：", serial_results)
print("串行耗时：", round(serial_elapsed, 2), "秒")

print("并发结果：", concurrent_results)
print("并发耗时：", round(concurrent_elapsed, 2), "秒")

模型 A 开始请求
模型 A 请求完成
模型 B 开始请求
模型 B 请求完成
模型 C 开始请求
模型 C 请求完成
模型 A 开始请求
模型 B 开始请求
模型 C 开始请求
模型 A 请求完成
模型 B 请求完成
模型 C 请求完成
串行结果： ['模型 A 的结果', '模型 B 的结果', '模型 C 的结果']
串行耗时： 3.03 秒
并发结果： ['模型 A 的结果', '模型 B 的结果', '模型 C 的结果']
并发耗时： 1.0 秒


## 错误练习

### 1. `SyntaxError`

**目标：** 主动制造一个语法错误，学会通过 traceback 的最后一行和 `^` 指示位置定位问题。

在新的代码格中运行以下错误代码：

```python
print("学习 Python"
```

然后运行下一个修复代码格，确认代码可以正常执行。

In [74]:
print("学习 Python"

SyntaxError: incomplete input (2377535772.py, line 1)

#### `SyntaxError` 修复代码

In [75]:
print("学习 Python")

学习 Python


### 2. `IndentationError`

先运行下面的错误代码，观察缩进错误；再运行修复代码。

In [76]:
def greet():
print("你好，Python")

IndentationError: expected an indented block after function definition on line 1 (3890415761.py, line 2)

#### `IndentationError` 修复代码

In [77]:
def greet():
    print("你好，Python")

greet()

你好，Python


### 3. `NameError`

错误代码使用了尚未定义的变量。

In [78]:
print(user_name)

小明


#### `NameError` 修复代码

In [79]:
user_name = "小明"
print(user_name)

小明


### 4. `TypeError`

错误代码直接使用 `+` 连接字符串和整数。

In [80]:
hours = 3
message = "预计学习 " + hours + " 小时"
print(message)

TypeError: can only concatenate str (not "int") to str

#### `TypeError` 修复代码

In [81]:
hours = 3
message = f"预计学习 {hours} 小时"
print(message)

预计学习 3 小时


### 5. `KeyError`

错误代码使用 `[]` 访问字典中不存在的 `model` 键。

In [82]:
config = {"temperature": 0.2}
print(config["model"])

KeyError: 'model'

#### `KeyError` 修复代码

In [83]:
config = {"temperature": 0.2}
model = config.get("model")

if model is None:
    print("配置缺少必填字段：model")
else:
    print(model)

配置缺少必填字段：model


### 6. `ModuleNotFoundError`

错误代码故意把 `pydantic` 拼错为 `pydanitc`。

In [84]:
import pydanitc

ModuleNotFoundError: No module named 'pydanitc'

#### `ModuleNotFoundError` 修复代码

In [85]:
from pydantic import BaseModel

print("pydantic 导入成功", BaseModel)

pydantic 导入成功 <class 'pydantic.main.BaseModel'>


### 7. 选错 Jupyter Kernel 导致导入失败

这个错误不是代码拼错，而是 Notebook 选了没有安装项目的 Python 环境。

1. 先在当前 `Python (llm-learning)` Kernel 中运行下方诊断格，记住解释器路径。
2. 在 Notebook 界面右上角把 Kernel 临时切换成另一个 Python 环境。
3. 重新运行导入格，观察 `ModuleNotFoundError: No module named 'study_app'`。
4. 切回 `Python (llm-learning)`，重启 Kernel，再次运行诊断格和导入格。

In [86]:
import importlib.util
import sys

print("Python 解释器：", sys.executable)
print("study_app 位置：", importlib.util.find_spec("study_app"))

Python 解释器： d:\develop\miniforge3\envs\llm-learning\python.exe
study_app 位置： None


#### Kernel 导入验证代码

在错误 Kernel 中运行时应该失败；切回 `llm-learning` 后运行应该成功。

In [1]:
from study_app.schemas import StudyTask

task = StudyTask(title="验证 Kernel", priority=1, hours=1)
print("导入成功：", task.model_dump())

导入成功： {'title': '验证 Kernel', 'priority': 1, 'hours': 1.0, 'completed': False}
